# Guía 4 - Gobernanza y ética de datos

In [2]:
#IMPORTACION DE LIBRERIAS

#Aqui se importan las librerias necesarias para manipular datos y las rutas

import pandas as pd
import sqlite3
import json
from pathlib import Path
import numpy as np
%pip install faker

print("Librerias importadas!!!!!!!!!!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.9 MB/s eta 0:00:00
Librerias importadas!!!!!!!!!!


In [3]:
# DEFINICION DE RUTAS DE TRABAJO

# Aqui se configuran las rutas para organizar los archivos del proyecto

BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR / "datos" / "crudos"
DATA_PROCESSED = BASE_DIR / "datos" / "procesados"
DATA_EXTERNAL = BASE_DIR / "datos" / "externos"
DOCUMENTOS = BASE_DIR / "documentos"

DOCUMENTOS.mkdir(parents=True, exist_ok=True)

print(f"Datos crudos: {DATA_RAW}")
print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Datos externos: {DATA_EXTERNAL}")
print(f"Documentos: {DOCUMENTOS}")
print("Rutas de trabajo definidas correctamente.")

Datos crudos: /content/datos/crudos
Datos procesados: /content/datos/procesados
Datos externos: /content/datos/externos
Documentos: /content/documentos
Rutas de trabajo definidas correctamente.


In [59]:
# GUÍA 4 - GOBERNANZA Y ÉTICA DE DATOS

In [60]:
# IMPORTACIÓN DE LIBRERÍAS

# Aquí importamos pandas y numpy para el manejo de los datos estructurados,
# hashlib para la encriptación (seudonimización) y pathlib para gestionar las rutas.
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

print("Librerias importadas-------------------")

Librerias importadas-------------------


In [61]:
# CREACIÓN DE DIRECTORIOS DE GOBERNANZA

# Aquí se configuran las rutas y se crean las carpetas necesarias para almacenar
# tanto los datos seguros (gobernanza) como los documentos legales y matrices.
BASE_DIR = Path("/content/")
DATA_GOV = BASE_DIR/"datos"/"gobernanza"
DOCS_DIR = BASE_DIR/"documentos"

DATA_GOV.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datos gobernanza: {DATA_GOV}")
print(f"Documentos: {DOCS_DIR}")
print("Carpetas de gobernanza creadas exitosamente-------------------")

Datos gobernanza: /content/datos/gobernanza
Documentos: /content/documentos
Carpetas de gobernanza creadas exitosamente-------------------


In [62]:
# CREACIÓN DE DATOS FICTICIOS PARA SMARTCAMPUS (10,000 ALUMNOS)

# Si faker no está instalado en el entorno, descomentar la siguiente línea:
# !pip install faker

from faker import Faker
import random

fake = Faker('es_ES')

N_ALUMNOS = 10000

catalogo_carreras = {
    "Ingenieria": ["Sistemas", "Industrial", "Electrica", "Mecatronica"],
    "Diseño": ["Animacion Digital", "Diseño Grafico", "Diseño de Interiores"],
    "Ciencias": ["Ciencias de la Computacion", "Matematica Aplicada"]
}

facultades_disponibles = list(catalogo_carreras.keys())

registros_estudiantes = []
for i in range(1, N_ALUMNOS + 1):
    fac = random.choice(facultades_disponibles)
    carr = random.choice(catalogo_carreras[fac])
    nombre_alumno = fake.name()

    # Sanitización simple para el correo institucional
    alias_correo = (
        nombre_alumno.lower()
        .replace(" ", ".")
        .replace("ñ", "n")
        .replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
    )[:18]

    registros_estudiantes.append({
        "id_estudiante": 20260000 + i,
        "nombre": nombre_alumno,
        "correo": f"{alias_correo}@campus.edu.sv",
        "telefono": f"7{random.randint(1000000, 9999999)}",
        "edad": int(np.random.choice([18, 19, 20, 21, 22, 23, 25, 29, 35, 42],
                                     p=[0.20, 0.25, 0.20, 0.15, 0.08, 0.05, 0.03, 0.02, 0.015, 0.005])),
        "facultad": fac,
        "carrera": carr
    })

estudiantes = pd.DataFrame(registros_estudiantes)

print(f"Datos ficticios de estudiantes generados: {len(estudiantes)} registros.")
print(estudiantes.head(3))

Datos ficticios de estudiantes generados: 10000 registros.
   id_estudiante                nombre                            correo  \
0       20260001    Chema Alberto Real  chema.alberto.real@campus.edu.sv   
1       20260002  Juan Pablo de Arenas  juan.pablo.de.aren@campus.edu.sv   
2       20260003    Martirio del Peiró  martirio.del.peiro@campus.edu.sv   

   telefono  edad    facultad         carrera  
0  74453250    20  Ingenieria      Industrial  
1  79359812    20      Diseño  Diseño Grafico  
2  75230568    18  Ingenieria        Sistemas  


In [63]:
# MATRIZ DE CLASIFICACIÓN DE DATOS

# Definimos el nivel de sensibilidad de cada columna basándonos en la normativa
# de protección de datos para establecer qué controles técnicos aplicar.
clasificacion = pd.DataFrame([
    ["id_estudiante", "Identificador interno", "Confidencial", "Vincular registros", "Seudonimizar"],
    ["nombre", "Identificador directo", "Confidencial", "Contacto", "Excluir de vista analítica"],
    ["correo", "Contacto", "Confidencial", "Contacto", "Enmascarar o excluir"],
    ["telefono", "Contacto", "Confidencial", "Contacto", "Enmascarar o excluir"],
    ["edad", "Demográfico", "Interno", "Segmentación", "Generalizar por rango"],
    ["facultad", "Ubicación académica", "Interno", "Análisis de uso", "Conservar"],
    ["carrera", "Académico", "Interno", "Analítica estudiantil", "Conservar"]
],
columns=["campo", "tipo", "clasificacion", "finalidad", "control"])

print("Matriz de clasificacion estructurada---------------")
print(clasificacion)

Matriz de clasificacion estructurada---------------
           campo                   tipo clasificacion              finalidad  \
0  id_estudiante  Identificador interno  Confidencial     Vincular registros   
1         nombre  Identificador directo  Confidencial               Contacto   
2         correo               Contacto  Confidencial               Contacto   
3       telefono               Contacto  Confidencial               Contacto   
4           edad            Demográfico       Interno           Segmentación   
5       facultad    Ubicación académica       Interno        Análisis de uso   
6        carrera              Académico       Interno  Analítica estudiantil   

                      control  
0                Seudonimizar  
1  Excluir de vista analítica  
2        Enmascarar o excluir  
3        Enmascarar o excluir  
4       Generalizar por rango  
5                   Conservar  
6                   Conservar  


In [64]:
# GUARDADO DE LA MATRIZ DE CLASIFICACIÓN

# Exportamos la matriz a un archivo Excel para que quede como evidencia
# documental de las políticas de gobierno de datos aplicadas.
ruta_clasificacion = DOCS_DIR/"clasificacion_datos_Smart_Campus.xlsx"
clasificacion.to_excel(ruta_clasificacion, index=False)
print(f"Clasificacion guardada en: {ruta_clasificacion}")

Clasificacion guardada en: /content/documentos/clasificacion_datos_Smart_Campus.xlsx


In [65]:
# SEUDONIMIZACIÓN DE IDENTIFICADORES

# Aplicamos una función hash SHA-256 junto con una "sal" (salt) criptográfica
# para convertir el carnet del estudiante en un token irreversible de 12 caracteres.
SALT = "smartcampus-datos-sinteticos"
def tokenizar(valor):
    """Convierte un valor en un codigo unico de 12 caracteres."""
    texto = f"{SALT}|{valor}".encode('utf-8')
    return hashlib.sha256(texto).hexdigest()[:12]

estudiantes["estudiante_token"] = estudiantes["id_estudiante"].astype(str).apply(tokenizar)

print("Seudonimizacion aplicada correctamente:-----------")
print(estudiantes[["id_estudiante", "estudiante_token"]])

Seudonimizacion aplicada correctamente:-----------
      id_estudiante estudiante_token
0          20260001     7a076f3716ed
1          20260002     2f83990be8fb
2          20260003     fbc23ad1ed9b
3          20260004     8ade8ef82f1f
4          20260005     5ed70503c4bd
...             ...              ...
9995       20269996     c019384f13cc
9996       20269997     dbb898fec953
9997       20269998     5403de44c5e1
9998       20269999     635a863637b3
9999       20270000     1d3a916ad10b

[10000 rows x 2 columns]


In [66]:
# ENMASCARAMIENTO DE CORREO ELECTRÓNICO

# Ocultamos la información sensible del correo dejando visible solo la primera
# letra del usuario y el dominio institucional para mantener el formato.
def enmascarar_correo(correo):
    """Muestra solo la primera letra del correo y el dominio completo"""
    usuario, dominio = correo.split('@', 1)
    visible = usuario[:1]
    return f"{visible}***@{dominio}"

estudiantes["correo_mask"] = estudiantes["correo"].apply(enmascarar_correo)
print("Correo enmascarado exitosamente:----------------")
print(estudiantes[["correo", "correo_mask"]])

Correo enmascarado exitosamente:----------------
                                correo         correo_mask
0     chema.alberto.real@campus.edu.sv  c***@campus.edu.sv
1     juan.pablo.de.aren@campus.edu.sv  j***@campus.edu.sv
2     martirio.del.peiro@campus.edu.sv  m***@campus.edu.sv
3     eleuterio.zaragoza@campus.edu.sv  e***@campus.edu.sv
4            ileana.goni@campus.edu.sv  i***@campus.edu.sv
...                                ...                 ...
9995  marisela.carreras.@campus.edu.sv  m***@campus.edu.sv
9996  alfonso.echevarria@campus.edu.sv  a***@campus.edu.sv
9997  sandra.pozo.valcar@campus.edu.sv  s***@campus.edu.sv
9998  delfina.esteve-mat@campus.edu.sv  d***@campus.edu.sv
9999  ivan.arellano.mend@campus.edu.sv  i***@campus.edu.sv

[10000 rows x 2 columns]


In [67]:
# ENMASCARAMIENTO DE NÚMERO TELEFÓNICO

# Ocultamos los primeros dígitos del teléfono y mostramos solo los últimos 4
# usando una función lambda rápida y efectiva.
estudiantes["telefono_mask"] = estudiantes["telefono"].astype(str).apply(lambda x: "****" + x[-4:])
print("Telefono enmascarado exitosamente:----------------")
print(estudiantes[["telefono", "telefono_mask"]])

Telefono enmascarado exitosamente:----------------
      telefono telefono_mask
0     74453250      ****3250
1     79359812      ****9812
2     75230568      ****0568
3     72499027      ****9027
4     79630994      ****0994
...        ...           ...
9995  72194957      ****4957
9996  78142712      ****2712
9997  77364562      ****4562
9998  76266684      ****6684
9999  78528470      ****8470

[10000 rows x 2 columns]


In [68]:
# GENERALIZACIÓN DE DATOS DEMOGRÁFICOS (EDAD)

# Convertimos la edad exacta en rangos (bins) para evitar la reidentificación
# de estudiantes que puedan tener edades atípicas en ciertas carreras.
bins = [0, 24, 34, 44, 54, 64, 200]
labels = ["<=24", "25-34", "35-44", "45-54", "55-64", "65+"]

estudiantes["rango_edad"] = pd.cut(estudiantes["edad"], bins=bins, labels=labels)
print("Edad generalizada por rangos:---------------")
print(estudiantes[["edad", "rango_edad"]])

Edad generalizada por rangos:---------------
      edad rango_edad
0       20       <=24
1       20       <=24
2       18       <=24
3       21       <=24
4       23       <=24
...    ...        ...
9995    20       <=24
9996    20       <=24
9997    18       <=24
9998    23       <=24
9999    20       <=24

[10000 rows x 2 columns]


In [69]:
# CREACIÓN DE VISTA ANALÍTICA PROTEGIDA (MINIMIZACIÓN)

# Aplicamos el principio de minimización de datos: extraemos únicamente las columnas
# que ya están protegidas y que son estrictamente necesarias para el análisis.
vista_analitica = estudiantes[[
    "estudiante_token",
    "rango_edad",
    "facultad",
    "carrera"
]].copy()

print("Vista analitica protegida (Sin identificadores directos):-----------")
print(vista_analitica)

Vista analitica protegida (Sin identificadores directos):-----------
     estudiante_token rango_edad    facultad                     carrera
0        7a076f3716ed       <=24  Ingenieria                  Industrial
1        2f83990be8fb       <=24      Diseño              Diseño Grafico
2        fbc23ad1ed9b       <=24  Ingenieria                    Sistemas
3        8ade8ef82f1f       <=24    Ciencias  Ciencias de la Computacion
4        5ed70503c4bd       <=24      Diseño           Animacion Digital
...               ...        ...         ...                         ...
9995     c019384f13cc       <=24      Diseño           Animacion Digital
9996     dbb898fec953       <=24    Ciencias         Matematica Aplicada
9997     5403de44c5e1       <=24      Diseño        Diseño de Interiores
9998     635a863637b3       <=24  Ingenieria                 Mecatronica
9999     1d3a916ad10b       <=24    Ciencias         Matematica Aplicada

[10000 rows x 4 columns]


In [70]:
# GUARDADO DEL DATASET PROTEGIDO

# Exportamos esta vista segura en formato CSV dentro de la carpeta de gobernanza.
# Este es el único archivo que debería compartirse con los analistas de datos.
ruta_publicable = DATA_GOV/"dataset_publicable_Smart_Campus.csv"
vista_analitica.to_csv(ruta_publicable, index=False, encoding="utf-8")
print(f"Vista protegida guardada en: {ruta_publicable}")

Vista protegida guardada en: /content/datos/gobernanza/dataset_publicable_Smart_Campus.csv


In [71]:
# VALIDACIÓN AUTOMÁTICA DE PRIVACIDAD

# Verificamos mediante código que por error humano no se haya incluido ninguna
# columna confidencial en la vista analítica final.
identificadores_directos = {"nombre", "correo", "telefono", "id_estudiante"}
expuestos = identificadores_directos.intersection(vista_analitica.columns)

print("Verificando identificadores directos:-------------------")
print(f"Identificadores directos expuestos: {list(expuestos)}")

if len(expuestos) == 0:
    print("VALIDACIÓN EXITOSA: Ningun identificador directo expuesto")
else:
    print("ALERTA CRÍTICA: ¡IDENTIFICADORES DIRECTOS EXPUESTOS!")

Verificando identificadores directos:-------------------
Identificadores directos expuestos: []
VALIDACIÓN EXITOSA: Ningun identificador directo expuesto


In [72]:
# MATRIZ DE RIESGOS DE DATOS

# Identificamos, priorizamos y asignamos controles a los posibles riesgos
# de privacidad en el proyecto SmartCampus. (Puntaje = Probabilidad × Impacto)
riesgos = pd.DataFrame([
    ["Exposición de identificadores directos", 4, 5, "Eliminar o enmascarar campos", "Líder de datos", "Dataset protegido"],
    ["Reidentificación por combinación de atributos", 3, 4, "Generalizar y revisar granularidad", "Analista", "Análisis de grupos"],
    ["Acceso excesivo al dataset crudo", 3, 5, "RBAC y mínimo privilegio", "Administrador", "Matriz de acceso"],
    ["Retención indefinida de copias", 3, 3, "Regla de retención", "Líder de proyecto", "Bitácora de eliminación"],
    ["Uso del dato para finalidad no definida", 2, 5, "Registrar propósito", "Propietario del dato", "Ficha de finalidad"],
    ["Conclusiones sesgadas por segmentación", 3, 4, "Revisión ética", "Equipo analítico", "Reflexión y pruebas"]
],
columns=["riesgo", "probabilidad", "impacto", "control", "responsable", "evidencia"])

riesgos["puntaje"] = riesgos["probabilidad"] * riesgos["impacto"]
riesgos["nivel"] = riesgos["puntaje"].apply(
    lambda x: "Crítico" if x >= 16 else "Alto" if x >= 10 else "Moderado" if x >= 5 else "Bajo"
)

print("MATRIZ DE RIESGOS GENERADA:")
print(riesgos.sort_values("puntaje", ascending=False))

MATRIZ DE RIESGOS GENERADA:
                                          riesgo  probabilidad  impacto  \
0         Exposición de identificadores directos             4        5   
2               Acceso excesivo al dataset crudo             3        5   
1  Reidentificación por combinación de atributos             3        4   
5         Conclusiones sesgadas por segmentación             3        4   
4        Uso del dato para finalidad no definida             2        5   
3                 Retención indefinida de copias             3        3   

                              control           responsable  \
0        Eliminar o enmascarar campos        Líder de datos   
2            RBAC y mínimo privilegio         Administrador   
1  Generalizar y revisar granularidad              Analista   
5                      Revisión ética      Equipo analítico   
4                 Registrar propósito  Propietario del dato   
3                  Regla de retención     Líder de proyecto   

    

In [73]:
# GUARDADO DE LA MATRIZ DE RIESGOS

# Persistimos la evaluación de riesgos en formato Excel para auditorías futuras.
ruta_riegos = DOCS_DIR/"matriz_riesgos_Smart_Campus.xlsx"
riesgos.to_excel(ruta_riegos, index=False)
print(f"Matriz de riesgos guardada en:----------------{ruta_riegos}")

Matriz de riesgos guardada en:----------------/content/documentos/matriz_riesgos_Smart_Campus.xlsx


In [74]:
# POLÍTICA BREVE DE GOBERNANZA - Reglas de uso, acceso y protección

politica = """
# Gobernanza de datos: Smart Campus Analytics

# 1. Objetivo
Establecer las reglas para recolectar, tratar, almacenar y publicar los datos del proyecto, de modo que los resultados sean confiables, verificables y respetuosos con la privacidad de las personas.

# 2. Roles y responsabilidades
- Propietario de los datos: la institución (coordinación académica y unidad de laboratorios), que autoriza el uso de la información.
- Analista de datos: limpia, transforma y analiza los datos, y documenta cada paso.
- Responsable de calidad: valida que los indicadores y las cifras coincidan con las fuentes.
- Usuarios del informe: dirección y coordinadores, con acceso solo a resultados agregados.

# 3. Fuentes y calidad de los datos
- Cada fuente (asistencia, ocupación de laboratorios, horarios, recursos tecnológicos) queda registrada con su origen, periodo y frecuencia de actualización.
- Se aplican controles básicos: revisión de valores nulos, duplicados, formatos de fecha y hora, y rangos válidos.
- Los criterios de limpieza (qué se elimina, qué se imputa y por qué) se documentan en el notebook.

# 4. Privacidad y ética
- Se trabaja con datos agregados por curso, laboratorio o franja horaria; no se identifica a estudiantes ni docentes de forma individual.
- Se eliminan o anonimizan nombres, correos, identificadores y cualquier dato personal antes del análisis.
- No se publican ubicaciones sensibles ni información que permita identificar a una persona.
- Los datos se usan únicamente para el fin académico del proyecto.

# 5. Seguridad y acceso
- El acceso a los datos originales se limita a los integrantes del equipo.
- Los archivos se almacenan en un repositorio compartido con permisos controlados, sin datos sensibles en el notebook público.
- Se conserva una copia de los datos crudos, sin modificar, separada de los datos procesados.

# 6. Trazabilidad y transparencia
- Se mantiene una **matriz de trazabilidad** que relaciona cada hallazgo con su fuente, indicador y visualización.
- Se documentan filtros, transformaciones y fórmulas de cada KPI, con control de versiones del notebook y de los conjuntos de datos.
- Las limitaciones (periodo corto, datos faltantes, baja granularidad) se declaran de forma explícita en el informe.

# 7. Ciclo de vida de los datos
1. Recolección: obtención de los datos con autorización.
2. Procesamiento: limpieza, transformación y validación.
3. Análisis y publicación: dashboard e informes con resultados agregados.
4. Conservación y eliminación: los datos se conservan mientras dure el proyecto y luego se eliminan o se archivan de forma anonimizada.

# 8. Revisión
El equipo revisará estas reglas al inicio de cada entrega para confirmar que se cumplen y actualizarlas si cambian las fuentes o el alcance del proyecto.
"""

In [75]:
# GUARDADO DE LA POLÍTICA

# Guardamos el archivo markdown generado con los lineamientos éticos.
ruta_politica = DOCS_DIR/"politica_gobernanza_Smart_Campus.md"
with open(ruta_politica, "w", encoding="utf-8") as f:
    f.write(politica)
print(f"Politica guardada exitosamente en: {ruta_politica}")

Politica guardada exitosamente en: /content/documentos/politica_gobernanza_Smart_Campus.md


In [76]:
# VERIFICACIÓN FINAL DE ARCHIVOS GENERADOS

# Listamos el contenido de los directorios para comprobar que todos los
# documentos y datasets se han guardado correctamente en su ubicación.
print("Verificando archivos de datos generados:------------------")
print("=" * 40)
!ls -la /content/datos/gobernanza/
print("Verificando documentos legales generados:------------------")
!ls -la /content/documentos/

Verificando archivos de datos generados:------------------
total 448
drwxr-xr-x 2 root root   4096 Oct 22 00:00 .
drwxr-xr-x 7 root root   4096 Oct 22 00:00 ..
-rw-r--r-- 1 root root 449394 Oct 22 00:00 dataset_publicable_Smart_Campus.csv
Verificando documentos legales generados:------------------
total 28
drwxr-xr-x 2 root root 4096 Oct 22 00:00 .
drwxr-xr-x 1 root root 4096 Oct 22 00:00 ..
-rw-r--r-- 1 root root 5339 Oct 22 00:00 clasificacion_datos_Smart_Campus.xlsx
-rw-r--r-- 1 root root 5595 Oct 22 00:00 matriz_riesgos_Smart_Campus.xlsx
-rw-r--r-- 1 root root 2814 Oct 22 00:00 politica_gobernanza_Smart_Campus.md
